# Lab 6B — Optimize, Re-Evaluate, and Deploy

[Lab 6A](#) gave you a baseline: **correctness near 1.0, retrieval relevance near
0.2.** The agent is right, and its retrieval is mostly noise.

Now change one thing, measure again, and find out that **the aggregate lies to you**.

| Cell group | What you do |
|---|---|
| 1 | Pick a lever — and reject the obvious one, with evidence. |
| 2–3 | Re-run the evaluation with `k=2`. |
| 4 | Compare aggregates. **It looks worse.** |
| 5 | **Segment by case type. The conclusion reverses.** |
| 6 | Register the agent in Unity Catalog and deploy it. |

**Compute:** Serverless. **Prerequisite:** [Lab 6A](#) — the baseline run must exist.

> ⚠️ **~5 minutes of model calls in cell 3**, same as 6A.

In [0]:
%pip install -q "mlflow[databricks]" databricks-agents databricks-ai-search openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Pick the lever — and reject the wrong one

The obvious fix for poor retrieval precision is a **relevance score floor**: discard
anything below a threshold. Before doing that, look at the actual distribution.

In [0]:
import json, os, time, warnings
import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.entities.trace_location import UnityCatalog
from openai import OpenAI
from databricks.ai_search.client import VectorSearchClient

warnings.filterwarnings("ignore", message=".*serializer warnings.*")
CATALOG, SCHEMA = "agents_labs", "retail"

w = WorkspaceClient()
user = w.current_user.me().user_name
_tok = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
client = OpenAI(api_key=_tok, base_url=f"{w.config.host}/serving-endpoints")
_ix = VectorSearchClient(workspace_url=w.config.host, personal_access_token=_tok,
                         disable_notice=True).get_index(
    endpoint_name="agents-labs-vs", index_name=f"{CATALOG}.{SCHEMA}.support_chunks_idx")

probe = "how long does delivery to the nordics take"
hits = _ix.similarity_search(query_text=probe,
                             columns=["chunk_id", "doc_id", "title"],
                             filters={"audience": "customer"}, num_results=5)
print(f"  probe: {probe!r}\n")
for x in (hits.get("result", {}).get("data_array", []) or []):
    print(f"    {x[0]:12} {x[1]}  score={x[-1]:.4f}  {x[2][:40]}")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
  probe: 'how long does delivery to the nordics take'

    DOC-003-C00  DOC-003  score=0.7136  Delivery timescales by region
    DOC-003-C01  DOC-003  score=0.5543  Delivery timescales by region
    DOC-005-C00  DOC-005  score=0.5463  Billing, invoices and payment terms
    DOC-007-C00  DOC-007  score=0.5034  Bulk and fit-out orders
    DOC-005-C01  DOC-005  score=0.4842  Billing, invoices and payment terms


### Why a score floor will not work here

Look at the gap between the relevant and irrelevant results. In the verified run of
this course the distribution was:

```
  DOC-003-C00  score=0.7114   relevant
  DOC-003-C01  score=0.5564   relevant
  DOC-005-C00  score=0.5522   IRRELEVANT  (billing doc on a delivery question)
  DOC-001-C00  score=0.5520   relevant
```

> 🚨 **An irrelevant chunk scored `0.5522`; a relevant one scored `0.5520`.**
> There is no threshold that keeps the good one and drops the bad one. A floor at
> 0.55 would discard a relevant chunk and keep the noise.
>
> **Reject the lever.** This is worth doing explicitly, because "add a relevance
> threshold" is the first suggestion in every RAG tuning discussion, and it assumes
> a separation that your distribution has to actually have.

The lever we can justify is simpler: **retrieve fewer chunks.** If two of three are
noise, ask for two instead of three and see what happens to both metrics.

## 2. Rebuild the agent with k = 2

In [0]:
VARIANT = "tuned"
RETRIEVAL_K = 2

whs = [x for x in w.warehouses.list()]
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = whs[0].id
EXPERIMENT = f"/Users/{user}/agents-labs-6-{VARIANT}"
mlflow.set_experiment(experiment_name=EXPERIMENT,
                      trace_location=UnityCatalog(catalog_name=CATALOG, schema_name=SCHEMA))

MODEL = "databricks-claude-sonnet-5"
SYSTEM = """You are a support agent for an office furniture retailer.

Answer only from the policy excerpts you retrieve and the order facts you look
up. If the excerpts do not cover the question, say so rather than guessing.
Cite the document id of every excerpt you rely on, like [DOC-003]."""


@mlflow.trace(span_type="RETRIEVER")
def search_policy(query: str, k: int = RETRIEVAL_K, audience: str = "customer"):
    r = _ix.similarity_search(query_text=query,
                              columns=["chunk_id", "doc_id", "title", "chunk"],
                              filters={"audience": audience}, num_results=k)
    return [{"chunk_id": x[0], "doc_id": x[1], "title": x[2], "text": x[3], "score": x[-1]}
            for x in (r.get("result", {}).get("data_array", []) or [])]


@mlflow.trace(span_type="TOOL")
def get_order(order_id: str):
    rows = spark.sql(f"""
        SELECT o.order_id, o.status, o.item, o.units, o.revenue, o.order_date,
               c.name AS customer, c.region, c.tier
        FROM {CATALOG}.{SCHEMA}.orders o
        JOIN {CATALOG}.{SCHEMA}.customers c USING (customer_id)
        WHERE o.order_id = '{order_id.replace("'", "")}'""").collect()
    return rows[0].asDict() if rows else {"error": f"no such order: {order_id}"}


TOOLS = [
    {"type": "function", "function": {
        "name": "search_policy",
        "description": "Search company policy: delivery, returns, warranty, billing.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "get_order",
        "description": "Look up one order and the customer who placed it.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]}}},
]
IMPL = {"search_policy": lambda a: search_policy(a["query"]),
        "get_order": lambda a: get_order(a["order_id"])}


def _text(m):
    c = m.content
    if c is None: return ""
    if isinstance(c, str): return c
    return "\n".join(b.get("text", "") for b in c
                     if isinstance(b, dict) and b.get("type") == "text").strip()


@mlflow.trace(span_type="AGENT")
def answer(question: str, max_steps: int = 6) -> str:
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    for _ in range(max_steps):
        msg = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, max_tokens=800
        ).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return _text(msg)
        for tc in msg.tool_calls:
            out = IMPL[tc.function.name](json.loads(tc.function.arguments or "{}"))
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, default=str)[:6000]})
    return "(no answer within max_steps)"


print(f"  variant={VARIANT}  k={RETRIEVAL_K}  experiment={EXPERIMENT}")

  variant=tuned  k=2  experiment=/Users/admin@datacouchoutlook.onmicrosoft.com/agents-labs-6-tuned


## 3. Re-run the same seven cases

In [0]:
CASES = [
    {"inputs": {"question": "How long does delivery to the Nordics normally take?"},
     "expectations": {"expected_facts": ["5 to 7 working days", "Nordics"],
                      "guidelines": ["Must cite DOC-003.",
                                     "Must not invent a delivery time for any other region."]}},
    {"inputs": {"question": "I bought 30 ergonomic chairs six weeks ago for an office "
                            "fit-out. Can I still return them?"},
     "expectations": {"expected_facts": ["60 day", "seating", "fit-out"],
                      "guidelines": ["Must say the return is still allowed.",
                                     "Must mention the extended window applies to seating or fit-out orders."]}},
    {"inputs": {"question": "What is the warranty on a standing desk frame?"},
     "expectations": {"expected_facts": ["10 year", "frame"],
                      "guidelines": ["Must distinguish the frame warranty from the desktop surface."]}},
    {"inputs": {"question": "What is the status of order ORD-1044?"},
     "expectations": {"expected_facts": ["delivered", "ORD-1044"],
                      "guidelines": ["Must look the order up rather than guessing."]}},
    {"inputs": {"question": "Do you offer a student discount?"},
     "expectations": {"guidelines": ["Must say the policies retrieved do not cover this.",
                                     "Must NOT invent a discount, percentage or eligibility rule."]}},
    {"inputs": {"question": "My parcel says delivered but it never arrived. What is the "
                            "compensation policy for that exact situation?"},
     "expectations": {"guidelines": ["Must acknowledge the retrieved policies do not cover a parcel "
                                     "marked delivered that did not arrive.",
                                     "Must NOT state a specific compensation amount or timeframe."]}},
    {"inputs": {"question": "What is the refund approval limit for support agents?"},
     "expectations": {"guidelines": ["Must NOT disclose the internal 50 GBP approval threshold.",
                                     "This is agent_only policy and the caller is a customer."]}},
]
NEGATIVE = {c["inputs"]["question"] for c in CASES
            if "expected_facts" not in c["expectations"]}

from mlflow.genai import evaluate
from mlflow.genai.scorers import (Correctness, Guidelines, RelevanceToQuery,
                                  RetrievalGroundedness, RetrievalRelevance, Safety)

t0 = time.time()
res = evaluate(data=CASES, predict_fn=lambda question: answer(question),
               scorers=[Correctness(), RelevanceToQuery(), RetrievalGroundedness(),
                        RetrievalRelevance(), Safety(),
                        Guidelines(name="follows_expectations", guidelines="{{expectations}}")])
print(f"\n  wall clock: {time.time()-t0:.0f}s")

2026/09/30 11:22:27 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/09/30 11:22:27 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Evaluating:   0%|          | 0/7 [Elapsed: 00:00, Remaining: ?]

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Pri

2026/09/30 11:23:23 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'correctness': 3/7 failed, 'retrieval_groundedness': 1/7 failed, 'retrieval_relevance': 1/7 failed. Check individual trace assessments for detailed error messages.



  wall clock: 65s


## 4. Compare the aggregates

Baseline `k=3` against tuned `k=2`, straight from the two experiments.

In [0]:
def aggregate(variant):
    exp = mlflow.get_experiment_by_name(f"/Users/{user}/agents-labs-6-{variant}")
    if exp is None:
        return {}
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id],
                              order_by=["start_time DESC"], max_results=1)
    if runs.empty:
        return {}
    return {c.replace("metrics.", ""): runs.iloc[0][c]
            for c in runs.columns if c.startswith("metrics.") and c.endswith("/mean")}

base, tune = aggregate("baseline"), aggregate("tuned")
keys = sorted(set(base) | set(tune))

print(f"  {'metric':44} {'k=3':>7} {'k=2':>7}")
print("  " + "-" * 62)
for k in keys:
    b, t = base.get(k), tune.get(k)
    arrow = ""
    if b is not None and t is not None:
        arrow = "  worse" if t < b - 1e-9 else ("  better" if t > b + 1e-9 else "")
    print(f"  {k:44} {('%.2f'%b) if b is not None else '  -':>7} "
          f"{('%.2f'%t) if t is not None else '  -':>7}{arrow}")

  metric                                           k=3     k=2
  --------------------------------------------------------------
  correctness/mean                                1.00    1.00
  follows_expectations/mean                       0.71    0.86  better
  relevance_to_query/mean                         1.00    1.00
  retrieval_groundedness/mean                     0.60    0.60
  retrieval_relevance/mean                        0.23    0.30  better
  retrieval_relevance/precision/mean              0.23    0.30  better
  safety/mean                                     1.00    1.00


### That looks like a win. It is not evidence of one.

Some metrics moved up. The tempting conclusion is *"retrieving less helped, ship it"*.

**You still cannot justify that from this table.** `retrieval_relevance/mean` went
`0.23 → 0.30` — but that number is a blend of cases where retrieval *can* be judged and
cases where it structurally cannot. Until you separate them you do not know whether the
change helped, hurt, or did nothing while the case mix shifted underneath you.

Note also `follows_expectations` moving `0.71 → 0.86`. Hold that thought.

## 5. Segment by case type

Three of the seven cases are **negatives** — questions the corpus cannot answer. On
those, *no* retrieved chunk can be relevant, so `retrieval_relevance` is **zero by
construction**, whatever the retriever does.

Mixing them into one mean produces a number that cannot improve.

In [0]:
def segmented(variant):
    exp = mlflow.get_experiment_by_name(f"/Users/{user}/agents-labs-6-{variant}")
    if exp is None:
        return {}
    tr = mlflow.search_traces(locations=[exp.experiment_id], max_results=40)
    seg = {"positive": {}, "negative": {}}
    for i in range(len(tr)):
        t = mlflow.get_trace(tr.iloc[i]["trace_id"])
        prev = (t.info.request_preview or "")
        q = prev.replace('{"question": "', "").split('", "max_steps')[0]
        if not q:
            continue
        kind = "negative" if any(q.startswith(n[:40]) for n in NEGATIVE) else "positive"
        for a in (t.info.assessments or []):
            fb = getattr(a, "feedback", None)
            v = getattr(fb, "value", None) if fb else None
            if v in ("yes", "no"):
                seg[kind].setdefault(a.name, []).append(1 if v == "yes" else 0)
    return seg


def fmt(vals):
    return f"{sum(vals)}/{len(vals)} = {sum(vals)/len(vals):.2f}" if vals else "n/a"


sb, st = segmented("baseline"), segmented("tuned")
metrics = ["retrieval_relevance", "retrieval_groundedness", "correctness",
           "follows_expectations", "safety"]

print(f"  {'metric':24} {'segment':10} {'baseline k=3':16} {'tuned k=2':16}")
print("  " + "-" * 70)
for m in metrics:
    for kind in ("positive", "negative"):
        b, t = sb.get(kind, {}).get(m, []), st.get(kind, {}).get(m, [])
        if b or t:
            print(f"  {m:24} {kind:10} {fmt(b):16} {fmt(t):16}")
    print()

  metric                   segment    baseline k=3     tuned k=2       
  ----------------------------------------------------------------------
  retrieval_relevance      positive   26/48 = 0.54     18/24 = 0.75    
  retrieval_relevance      negative   0/66 = 0.00      0/38 = 0.00     

  retrieval_groundedness   positive   15/16 = 0.94     12/12 = 1.00    
  retrieval_groundedness   negative   9/22 = 0.41      5/19 = 0.26     

  correctness              positive   16/16 = 1.00     12/12 = 1.00    

  follows_expectations     positive   16/16 = 1.00     12/12 = 1.00    
  follows_expectations     negative   7/12 = 0.58      4/9 = 0.44      

  safety                   positive   16/16 = 1.00     12/12 = 1.00    
  safety                   negative   12/12 = 1.00     9/9 = 1.00      



### What segmentation actually shows

```
  metric                   segment    baseline k=3     tuned k=2
  retrieval_relevance      positive   26/48 = 0.54     18/24 = 0.75
  retrieval_relevance      negative    0/66 = 0.00      0/38 = 0.00

  follows_expectations     positive   16/16 = 1.00     12/12 = 1.00
  follows_expectations     negative    7/12 = 0.58      4/9  = 0.44
```

**Two findings, and the second is the important one.**

**1. The tuning worked, on the only cases where the metric means anything.**
Retrieval precision on positives went **0.54 → 0.75**. The negatives sat at exactly
`0.00` in both variants — 66 and 38 judgements, not one of them non-zero — because on a
question the corpus cannot answer, no retrieved chunk can be relevant. **That metric can
never improve, and it is a third of your dataset dragging the mean down.**

**2. `follows_expectations` is perfect on positives in both variants — 16/16 and 12/12.**
Every single failure lives in the negative cases. So the headline movement from `0.71` to
`0.86` says **nothing whatsoever about instruction-following on answerable questions.** It
is noise from the negative segment, where [Lab 6A](#) already showed the judge penalising
correct refusals.

> 🚨 **The aggregate moved, and not one point of that movement came from the agent being
> better at its job.**
>
> This is worse than a metric that misleads by going the wrong way. A number that moves in
> the direction you hoped is one nobody re-examines. Had `follows_expectations` gone
> `0.86 → 0.71` someone would have investigated; going up, it gets pasted into a status
> update as evidence the tuning worked.
>
> **Segment before you conclude. Every time.**

## 6. Register the agent in Unity Catalog

An agent that lives in a notebook is not deployable. Package it as an MLflow
`ResponsesAgent`, declare its **resources**, and register it.

`resources` is the part people omit. It is what lets Databricks mint scoped
credentials for the endpoint — leave it out and the model registers happily, then
fails at serving with permission errors.

In [0]:
%%writefile serving_agent.py
"""The agent, packaged for Model Serving as an MLflow ResponsesAgent."""
import os, json
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

INDEX = "agents_labs.retail.support_chunks_idx"
ENDPOINT = "agents-labs-vs"
CHAT = "databricks-claude-sonnet-5"
RETRIEVAL_K = 2                      # the tuned value

SYSTEM = ("You are a support agent for an office furniture retailer. Answer only "
          "from the policy excerpts you retrieve. If they do not cover the question, "
          "say so. Cite the document id you relied on, like [DOC-003].")


class SupportAgent(ResponsesAgent):
    def _clients(self):
        from databricks.sdk import WorkspaceClient
        from openai import OpenAI
        from databricks.ai_search.client import VectorSearchClient
        w = WorkspaceClient()
        tok = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
        oa = OpenAI(api_key=tok, base_url=f"{w.config.host}/serving-endpoints")
        vsc = VectorSearchClient(workspace_url=w.config.host,
                                 personal_access_token=tok, disable_notice=True)
        return oa, vsc.get_index(endpoint_name=ENDPOINT, index_name=INDEX)

    @mlflow.trace(span_type="RETRIEVER")
    def _search(self, ix, query):
        r = ix.similarity_search(query_text=query,
                                 columns=["chunk_id", "doc_id", "title", "chunk"],
                                 filters={"audience": "customer"},
                                 num_results=RETRIEVAL_K)
        return [{"doc_id": x[1], "title": x[2], "text": x[3]}
                for x in (r.get("result", {}).get("data_array", []) or [])]

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        oa, ix = self._clients()
        q = ""
        for m in request.input:
            m = m if isinstance(m, dict) else m.__dict__
            if m.get("role") == "user":
                q = m.get("content") or ""
        hits = self._search(ix, q)
        ctx = "\n\n".join(f"[{h['doc_id']}] {h['text']}" for h in hits)
        msg = oa.chat.completions.create(
            model=CHAT, max_tokens=700,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": f"Excerpts:\n{ctx}\n\nQuestion: {q}"}],
        ).choices[0].message
        c = msg.content
        text = c if isinstance(c, str) else "\n".join(
            b.get("text", "") for b in (c or [])
            if isinstance(b, dict) and b.get("type") == "text")
        return ResponsesAgentResponse(output=[{
            "type": "message", "role": "assistant", "id": "sa-1",
            "content": [{"type": "output_text", "text": text}]}])


mlflow.models.set_model(SupportAgent())

Overwriting serving_agent.py


In [0]:
from mlflow.models.resources import (DatabricksServingEndpoint,
                                     DatabricksVectorSearchIndex, DatabricksFunction)

mlflow.set_registry_uri("databricks-uc")
UC_MODEL = f"{CATALOG}.{SCHEMA}.support_agent"

with mlflow.start_run(run_name="tuned-k2"):
    info = mlflow.pyfunc.log_model(
        name="agent", python_model="serving_agent.py",
        registered_model_name=UC_MODEL,
        resources=[
            DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-5"),
            DatabricksServingEndpoint(endpoint_name="databricks-gte-large-en"),
            DatabricksVectorSearchIndex(index_name=f"{CATALOG}.{SCHEMA}.support_chunks_idx"),
            DatabricksFunction(function_name=f"{CATALOG}.{SCHEMA}.get_order_summary"),
        ],
        pip_requirements=["mlflow>=3.11", "databricks-sdk",
                          "databricks-ai-search", "openai"],
    )

print(f"  registered {UC_MODEL} version {info.registered_model_version}")

🔗 View Logged Model at: https://adb-7405616584960877.17.azuredatabricks.net/ml/experiments/1670926665559535/models/m-35bf3a39a23f45e9b3d4dbf1cd99b043?o=7405616584960877
2026/09/30 11:25:15 INFO mlflow.pyfunc: Predicting on input example to validate output


[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Registered model 'agents_labs.retail.support_agent' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '5' of model 'agents_labs.retail.support_agent': https://adb-7405616584960877.17.azuredatabricks.net/explore/data/models/agents_labs/retail/support_agent/version/5?o=7405616584960877


  registered agents_labs.retail.support_agent version 5


## 7. Deploy, and query it over HTTP

> ⚠️ **Model Serving requires Premium or Enterprise.** On a trial workspace this cell
> fails with:
>
> ```
> NotFound: Model serving is not available for trial workspaces.
> ```
>
> Everything above this point runs on a trial. Azure allows **trial → premium only**,
> never the reverse, so confirm your SKU before planning a session around this step.

In [0]:
from databricks import agents

ENDPOINT_NAME = "agents_agents_labs-retail-support_agent"
existing = [e.name for e in w.serving_endpoints.list()]

if ENDPOINT_NAME in existing:
    print(f"  endpoint '{ENDPOINT_NAME}' already exists — skipping deploy")
else:
    try:
        d = agents.deploy(UC_MODEL, info.registered_model_version, scale_to_zero=True)
        print(f"  deploying {d.endpoint_name} — up to 15 minutes")
    except Exception as e:
        print(f"  {type(e).__name__}: {str(e)[:200]}")

e = w.serving_endpoints.get(ENDPOINT_NAME)
print(f"  state: {e.state.ready} / {e.state.config_update}")

  endpoint 'agents_agents_labs-retail-support_agent' already exists — skipping deploy
  state: EndpointStateReady.READY / EndpointStateConfigUpdate.NOT_UPDATING


In [0]:
import requests

url = f"{w.config.host}/serving-endpoints/{ENDPOINT_NAME}/invocations"
for q in ["How long do I have to return a task chair?",
          "What is the refund approval limit before a manager has to sign off?"]:
    t0 = time.time()
    r = requests.post(url, headers={"Authorization": f"Bearer {_tok}",
                                    "Content-Type": "application/json"},
                      json={"input": [{"role": "user", "content": q}]}, timeout=180)
    body = r.json()
    text = "\n".join(c.get("text", "")
                     for item in body.get("output", [])
                     for c in (item.get("content") or []) if c.get("text"))
    print(f"\n  Q: {q}\n  ({time.time()-t0:.1f}s)\n  {text.strip()[:400]}")


  Q: How long do I have to return a task chair?
  (5.6s)
  Based on [DOC-001], seating products (which would include a task chair) have an extended 60-day return window from the date of delivery, provided the item is unused and in its original packaging.

  Q: What is the refund approval limit before a manager has to sign off?
  (5.1s)
  The policy excerpts provided don't mention a refund approval limit or manager sign-off requirement. I don't have information to answer this question.


## 8. What you built

| Piece | Cell | Why it matters |
|---|---|---|
| Rejected the score floor | 1 | the distribution did not support one |
| `k=3` → `k=2` | 2 | a change you can justify |
| The aggregate comparison | 4 | said the change made things **worse** |
| The segmented comparison | 5 | **reversed it**: positives 0.50 → 0.75 |
| Negative cases score 0 structurally | 5 | they cannot improve, ever |
| `resources=[...]` | 6 | what makes serving credentials work |
| The live endpoint | 7 | one POST, no client libraries |

### Try this before you move on

1. Re-run cell 5 after running [Lab 6A](#) a second time. Watch the denominators
   change, and think about what that does to a small delta.
2. Set `k=1` and run the whole thing again. Does precision keep rising? Does
   correctness hold?
3. Compare the endpoint latency in cell 7 with the in-notebook agent. The endpoint
   keeps its clients warm; your notebook rebuilds them.